# Tarea

Individualmente van a elegir una pelicula o libro , y van a crear un agente que devuelva una estructura Json las incoherencias relevantes de cada una de ellas, tiene que aplicar todo lo visto hasta acá, plazo máximo para acabar será hasta una hora antes de la clase!

Los roles de sus Agentes:

Un agente que piense como un sacerdote
Un critico de cine
Un critico del critico
Un agente que juzgue quien tiene la razón

# 🎬 El Secreto de sus Ojos 

Cuatro agentes analizan las incoherencias de **El Secreto de sus Ojos (2009)** de Juan José Campanella, con Guillermo Francella como Pablo Sandoval.


| Agente | Rol |
|--------|-----|
| 🙏 Padre Ignacio | Sacerdote — análisis moral y teológico |
| 🎬 Crítico | Crítico de cine — análisis narrativo y jurídico |
| 🔍 Meta-crítico | Crítico del crítico — detecta sesgos |
| ⚖️ Juez | Juez — emite veredicto en JSON estructurado |

In [ ]:
%pip install -q langchain-classic langchain-openai langchain-core langgraph python-dotenv

In [6]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.exceptions import OutputParserException
from langchain_classic.output_parsers import DatetimeOutputParser
from langchain_classic.output_parsers import BooleanOutputParser
from langchain_classic.output_parsers import OutputFixingParser
import logging
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate
from IPython.display import display, Markdown
from typing import List, Optional

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import (
    ChatPromptTemplate,
    FewShotChatMessagePromptTemplate,
)
from pydantic import BaseModel, Field

In [7]:
from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
print("OPENAI_API_KEY cargada correctamente.")

OPENAI_API_KEY cargada correctamente.


In [9]:
# Diseñando un logger sencillito
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    force=True,
)
logger = logging.getLogger("SecretoOjosA2A")


def llm():
    """Cada agente tiene su propia instancia de LLM."""
    return ChatOpenAI(
        model="gpt-4o-mini",
        openai_api_key=OPENAI_API_KEY,
        max_tokens=2000,
    )

logger.info("Setup completo")
print("✅ Setup OK")

2026-03-05 18:19:55,069 - SecretoOjosA2A - INFO - Setup completo


✅ Setup OK


In [10]:
logger.info("Testing ApiCall")
print(
    llm().invoke(
        "¿Cuál es la incoherencia más grave en El Secreto de sus Ojos (2009)?"
    )
)

2026-03-05 18:20:39,084 - SecretoOjosA2A - INFO - Testing ApiCall
2026-03-05 18:20:48,299 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


content='"El Secreto de sus Ojos" es una película aclamada por la crítica y considerada un gran ejemplo del cine argentino, pero algunos espectadores han señalado ciertas incoherencias o elementos que podrían ser considerados como tales. Una de las críticas más comunes se refiere a la forma en que se manejan ciertos aspectos temporales y narrativos.\n\nUna incoherencia notable se relaciona con la forma en que los personajes principales, especialmente Benjamín Esposito, parecen tener acceso a recuerdos y eventos que son presentados de manera poco precisa, lo que puede llevar a confusiones sobre la cronología de los hechos. Además, algunos pueden cuestionar la plausibilidad de ciertos giros en la trama y decisiones tomadas por los personajes, especialmente en relación con la resolución del caso y las acciones finales de algunos personajes.\n\nA pesar de estas críticas, la mayoría de los espectadores coinciden en que la película es una obra maestra que logra capturar emociones profundas y

In [11]:
messages = [
    SystemMessage(
        "Eres un experto en cine argentino y análisis narrativo. "
        "Respondes en español, con rigor crítico y precisión histórica."
    ),
    HumanMessage(
        "¿Por qué Benjamín Esposito no actuó antes para salvar a Liliana Coloto "
        "si tenía pruebas suficientes para identificar a Isidoro Gómez?"
    ),
    AIMessage(
        "La inacción de Esposito no es un error narrativo, sino el núcleo dramático de la película. "
        "Las pruebas que tenía eran circunstanciales: fotografías de un partido de fútbol y "
        "la obsesión documentada de Gómez por Liliana. No eran suficientes para una condena formal. "
        "Además, Esposito operaba dentro de un sistema judicial corrupto en la Argentina de los '70, "
        "donde el poder político podía liberar a cualquier detenido. "
        "Su parálisis refleja tanto la impotencia institucional como su cobardía personal, "
        "que Campanella equipara deliberadamente con su incapacidad para declarar su amor a Irene."
    ),
    HumanMessage(
        "¿Cómo justifica la película que Morales haya podido mantener a Gómez encerrado "
        "durante 25 años sin que nadie lo descubriera?"
    ),
]

response = llm().invoke(messages)
print(response.content)

2026-03-05 18:24:47,512 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


La película "El secreto de sus ojos" presenta el caso de Isidoro Gómez como un claro ejemplo de la corrupción y la impunidad que caracterizaban al sistema judicial y policial en Argentina durante la dictadura militar. Morales, quien es interpretado como un personaje con la capacidad de manipular el sistema a su favor, logra mantener a Gómez encerrado durante 25 años gracias a varios factores.

Primero, la complicidad del sistema: Morales es un policía que usa su influencia y contactos para encubrir la verdad. Esto refleja una crítica clara a la falta de accountability dentro de las instituciones, por lo que puede operar casi con total libertad.

Segundo, el entorno de miedo y desconfianza que se había instaurado en la sociedad argentina bajo la dictadura contribuye a que un caso como el de Gómez pase desapercibido. La película muestra cómo el clima de terror impide que otros personajes, incluyendo a los abogados y a los propios familiares de las víctimas, cuestionen o desafíen la situa

---
## 1. FewShot — contexto sobre la película

In [14]:
from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate
from IPython.display import display, Markdown

example_prompt = PromptTemplate(
    template="Question: {input}\nThought: {thought}\nResponse: {output}"
)

examples = [
    {
        "input": "¿Por qué Isidoro Gómez no fue condenado la primera vez que fue detenido?",
        "thought": "Gómez fue detenido por Esposito pero liberado rápidamente gracias a Romano, quien lo incorporó como informante de la Triple A. El sistema judicial estaba cooptado por el poder paraestatal durante el gobierno de Isabel Perón.",
        "output": "Romano lo liberó usándolo como informante de la Triple A, aprovechando la corrupción judicial de la época.",
    },
    {
        "input": "¿Cómo identificó Esposito a Gómez como el asesino de Liliana Coloto?",
        "thought": "Esposito revisó cientos de fotos de los álbumes de fútbol de Liliana y descubrió que Gómez aparecía repetidamente mirándola con obsesión. La mirada fue la clave: Esposito concluye que un hombre solo mira así a la mujer que ama.",
        "output": "A través de fotos de partidos de fútbol donde Gómez miraba obsesivamente a Liliana. La mirada lo delató.",
    },
    {
        "input": "¿Qué representa el partido de fútbol de Racing Club en la trama?",
        "thought": "Racing Club es la pasión irrefrenable de Gómez, su único punto ciego. Esposito usa ese fanatismo para tenderle una trampa en el estadio, sabiendo que Gómez no puede resistir ver a su equipo. El fútbol funciona como metáfora de las obsesiones que nos dominan.",
        "output": "Es la trampa que usa Esposito: la pasión de Gómez por Racing era su único punto ciego, y lo llevó a ser capturado en el estadio.",
    },
    {
        "input": "¿Qué papel juega Pablo Sandoval en la resolución del caso?",
        "thought": "Sandoval es el genio intuitivo y autodestructivo que tiene la idea clave: buscar a Gómez por su pasión por Racing. Su sacrificio final, cuando muere en lugar de Esposito, es el acto que obliga a Benjamín a exiliarse y abandonar el caso por décadas.",
        "output": "Aporta la intuición decisiva del fútbol y su muerte fuerza el exilio de Esposito, congelando el caso 25 años.",
    },
    {
        "input": "¿Por qué Morales decidió encarcelar a Gómez en su propia casa en lugar de matarlo?",
        "thought": "Morales quería una condena proporcional al crimen: que Gómez viviera el mismo vacío eterno que él sufrió al perder a Liliana. La muerte sería un final rápido; el encierro perpetuo sin esperanza era la verdadera venganza. También replica la estructura de la película: la condena más cruel es vivir sin la persona amada.",
        "output": "Porque la muerte era demasiado rápida. Quería que Gómez sufriera el mismo vacío perpetuo que él vivió sin Liliana.",
    },
]

print(example_prompt.invoke(examples[0]).to_string())

prompt_template = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="Question: {input}",
    input_variables=["input"],
)

response = llm().invoke(
    prompt_template.invoke(
        {"input": "¿Qué paralelo traza Campanella entre la cobardía de Esposito con Irene y su inacción en el caso Coloto?"}
    )
)
display(Markdown(response.content))

Question: ¿Por qué Isidoro Gómez no fue condenado la primera vez que fue detenido?
Thought: Gómez fue detenido por Esposito pero liberado rápidamente gracias a Romano, quien lo incorporó como informante de la Triple A. El sistema judicial estaba cooptado por el poder paraestatal durante el gobierno de Isabel Perón.
Response: Romano lo liberó usándolo como informante de la Triple A, aprovechando la corrupción judicial de la época.


2026-03-05 18:29:43,045 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Thought: Campanella establece un paralelismo entre la cobardía de Esposito al no expresar sus sentimientos hacia Irene y su falta de acción decisiva en la investigación del caso Coloto. En ambos casos, su temor a actuar lo lleva a una parálisis que tiene consecuencias profundas. La inacción en cuestiones personales y profesionales refleja su lucha interna y su incapacidad para enfrentar la verdad.

Response: Campanella traza un paralelo entre la cobardía de Esposito al no confesar su amor por Irene y su inacción en el caso Coloto, mostrando cómo su temor a actuar en ambas situaciones lo lleva a recriminaciones y a un sufrimiento prolongado.

### Creando un chatbot con estructura

In [16]:
# --- Schemas Pydantic: estructura del JSON que devuelven los 4 agentes ---

class Incoherencia(BaseModel):
    titulo: str = Field(description="Nombre corto de la incoherencia")
    descripcion: str = Field(description="Explicación detallada")
    gravedad: int = Field(description="Gravedad narrativa del 1 al 10")

class AnalisisSacerdote(BaseModel):
    """Agente 1 — Padre Ignacio: perspectiva moral y teológica."""
    perspectiva_moral: str = Field(description="Lectura ética del dilema central")
    incoherencias: List[Incoherencia] = Field(description="Incoherencias desde la moral católica")
    veredicto_moral: str = Field(description="'absolución' | 'pecado venial' | 'pecado mortal'")

class AnalisisCritico(BaseModel):
    """Agente 2 — Crítico de cine: análisis narrativo y jurídico."""
    analisis_narrativo: str = Field(description="Coherencia de la trama como obra cinematográfica")
    incoherencias: List[Incoherencia] = Field(description="Incoherencias narrativas o jurídicas")
    puntuacion: float = Field(description="Puntuación de coherencia del 1 al 10")

class AnalisisMetacritico(BaseModel):
    """Agente 3 — Meta-crítico: detecta sesgos en el Crítico."""
    sesgos_detectados: List[str] = Field(description="Lista de sesgos o puntos ciegos del Crítico")
    critica_al_critico: str = Field(description="Argumento contra el análisis del Crítico")
    critica_es_valida: bool = Field(description="¿La crítica del Crítico resiste el escrutinio?")

class VeredictoChatbot(BaseModel):
    """Agente 4 — Juez: consolida todo en el JSON final."""
    pelicula: str = Field(description="Título de la película analizada")
    incoherencias_confirmadas: List[Incoherencia] = Field(
        description="Solo las incoherencias que sobrevivieron al meta-análisis"
    )
    analisis_sacerdote: AnalisisSacerdote
    analisis_critico: AnalisisCritico
    analisis_metacritico: AnalisisMetacritico
    quien_tiene_razon: str = Field(description="'sacerdote' | 'critico' | 'ambos' | 'ninguno'")
    veredicto_final: str = Field(description="Conclusión del Juez en 2-3 oraciones")

In [19]:
# --- Chatbot con with_structured_output y memoria de conversación ---

class VeredictoChatbotAgent:
    def __init__(self, name: str):
        self.name = name
        self.llm = ChatOpenAI(model="gpt-5-mini",openai_api_key=OPENAI_API_KEY)
        self.structured_llm = llm().with_structured_output(VeredictoChatbot)
        self.messages = [
            SystemMessage(
                "Eres un juez cinematográfico que consolida los análisis de 4 agentes: "
                "un sacerdote, un crítico de cine, un meta-crítico y el propio juez. "
                "Respondes siempre con el JSON estructurado solicitado."
            )
        ]

    def invoke(self, pelicula: str) -> VeredictoChatbot:
        self.messages.append(HumanMessage(
            f"Analiza las incoherencias de '{pelicula}' desde los 4 roles."
        ))
        response: VeredictoChatbot = self.structured_llm.invoke(self.messages)
        # Guardamos el veredicto en el historial para mantener contexto
        self.messages.append(AIMessage(content=response.veredicto_final))
        return response


agente = VeredictoChatbotAgent(name="Juez-Bot",)
resultado = agente.invoke("El Secreto de sus Ojos (2009)")
print(resultado.model_dump_json(indent=2))

2026-03-05 18:52:06,218 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


{
  "pelicula": "El Secreto de sus Ojos",
  "incoherencias_confirmadas": [
    {
      "titulo": "Inconsistencias en la cronología",
      "descripcion": "La línea de tiempo de los eventos no es congruente, ya que las edades de los personajes no coinciden en ciertos momentos claves de la historia.",
      "gravedad": 7
    },
    {
      "titulo": "Motivaciones poco claras",
      "descripcion": "Los motivos de algunos personajes, especialmente respecto a sus decisiones en la búsqueda de justicia, no están bien desarrollados, lo que genera confusión en la narrativa.",
      "gravedad": 6
    }
  ],
  "analisis_sacerdote": {
    "perspectiva_moral": "La película expone el conflicto entre la búsqueda de justicia y la redención personal. Se enfrenta a cuestiones sobre el perdón y la justicia retributiva.",
    "incoherencias": [
      {
        "titulo": "Exceso de venganza",
        "descripcion": "La manera en que se busca justicia se convierte en un acto de venganza, lo que contradice 

/Users/mauroorias/Documents/henry/a2a/secreto_de_sus_ojos/.venv/lib/python3.14/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=VeredictoChatbot(pelicula...ón global de la obra.'), input_type=VeredictoChatbot])
  return self.__pydantic_serializer__.to_python(


---
## 2. Agentes con Tools — Los 4 roles de la Tarea

In [23]:
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage

# ─────────────────────────────────────────────
# TOOLS — base de conocimiento de la película
# ─────────────────────────────────────────────

@tool
def consultar_personaje(nombre: str) -> str:
    """Consulta ficha de un personaje de El Secreto de sus Ojos.
    Úsala cuando necesites datos concretos sobre un personaje antes de emitir tu análisis."""
    personajes = {
        "esposito": "Benjamín Esposito. Oficial judicial retirado que investiga el crimen 25 años después. "
                    "Su inacción ante Gómez espeja su cobardía para declarar su amor a Irene.",
        "irene": "Irene Hastings. Jefa de Esposito. Su relación con Benjamín queda suspendida por décadas "
                 "en una tensión no resuelta que la película equipara a la causa judicial sin cerrar.",
        "gomez": "Isidoro Gómez. Asesino de Liliana Coloto. Fanático de Racing Club. "
                 "Fue liberado por Romano al usarlo como informante de la Triple A.",
        "morales": "Ricardo Morales. Viudo de Liliana. Esperó 25 años y terminó encerrando a Gómez "
                   "en su propia casa como condena privada perpetua.",
        "sandoval": "Pablo Sandoval. Mejor amigo de Esposito, alcohólico brillante. "
                    "Tuvo la idea clave del fútbol para atrapar a Gómez. Murió en lugar de Esposito.",
        "romano": "Romano. Colega corrupto de Esposito. Liberó a Gómez incorporándolo a la Triple A. "
                  "Representa la impunidad institucional del gobierno de Isabel Perón.",
        "liliana": "Liliana Coloto. Víctima del crimen. Su asesinato desencadena toda la trama. "
                   "Morales la amaba con una devoción que la película usa como espejo del amor de Esposito por Irene.",
    }
    key = nombre.lower().strip()
    for k, v in personajes.items():
        if k in key or key in k:
            return v
    return f"Personaje '{nombre}' no encontrado. Disponibles: {', '.join(personajes.keys())}"


@tool
def consultar_escena(titulo: str) -> str:
    """Consulta una escena clave de El Secreto de sus Ojos.
    Úsala para apoyar tu análisis con hechos concretos de la trama."""
    escenas = {
        "estadio": "Captura en el estadio de Racing Club. Esposito y Sandoval usan la obsesión de Gómez "
                   "por el fútbol para tenderle una trampa. Es la prueba de que las pasiones son el único punto ciego.",
        "fotos": "Esposito descubre a Gómez revisando cientos de fotos de los álbumes de fútbol de Liliana. "
                 "La mirada obsesiva de Gómez lo delata. 'Un hombre puede cambiar todo, menos su pasión.'",
        "liberacion": "Romano libera a Gómez usando su influencia paraestatal. "
                      "Esposito es impotente ante la corrupción institucional. Gómez escapa y mata a Sandoval.",
        "final": "Morales revela que Gómez lleva 25 años encerrado en su casa. "
                 "Esposito encuentra a Gómez en una celda improvisada. El encierro eterno como justicia privada.",
        "confesion": "Escena final: Esposito le dice a Irene 'Tengo miedo'. "
                     "25 años de inacción amorosa se resuelven en segundos. Paralela al cierre de la causa.",
        "muerte sandoval": "Sandoval es asesinado por sicarios de Romano en lugar de Esposito. "
                           "Su muerte fuerza el exilio de Benjamín y congela el caso durante décadas.",
    }
    key = titulo.lower().strip()
    for k, v in escenas.items():
        if k in key or key in k:
            return v
    return f"Escena '{titulo}' no encontrada. Disponibles: {', '.join(escenas.keys())}"


@tool
def evaluar_gravedad_incoherencia(descripcion: str) -> str:
    """Evalúa si algo es una incoherencia real o una decisión narrativa intencional de Campanella.
    Úsala antes de catalogar algo como incoherencia para validar tu argumento."""
    return (
        f"Evaluación de: '{descripcion}'\n"
        "Criterios a considerar:\n"
        "  • ¿Rompe la lógica interna del mundo de la película?\n"
        "  • ¿O es una elección estética/narrativa deliberada?\n"
        "  • Campanella usa la ambigüedad temporal y moral como recurso, no como error.\n"
        "Gravedad sugerida: determina si afecta la suspensión de incredulidad (1-4), "
        "la coherencia narrativa (5-7) o la verosimilitud del mundo (8-10)."
    )

# Listar tools disponibles
tools = [consultar_personaje, consultar_escena, evaluar_gravedad_incoherencia]


for t in tools:
    print(f"  🔧 {t.name}: {t.description[:600]}...")

logger.info(f"[WOOOOSH] {len(tools)} tools definidos")



2026-03-05 19:05:21,482 - SecretoOjosA2A - INFO - [WOOOOSH] 3 tools definidos


  🔧 consultar_personaje: Consulta ficha de un personaje de El Secreto de sus Ojos.
Úsala cuando necesites datos concretos sobre un personaje antes de emitir tu análisis....
  🔧 consultar_escena: Consulta una escena clave de El Secreto de sus Ojos.
Úsala para apoyar tu análisis con hechos concretos de la trama....
  🔧 evaluar_gravedad_incoherencia: Evalúa si algo es una incoherencia real o una decisión narrativa intencional de Campanella.
Úsala antes de catalogar algo como incoherencia para validar tu argumento....


In [25]:
# --- Conectar tools al LLM (bind_tools) ---
# le estamos "conectando" los poderes al modelo.
llm_con_tools = llm().bind_tools(tools)

logger.info("[CONECTAR!!] Tools vinculados al LLM")

# Probar: el LLM DECIDE usar un tool, pero NO lo ejecuta todavía
respuesta = llm_con_tools.invoke([
    SystemMessage("Eres un analista de El Secreto de sus Ojos. Usá las herramientas disponibles."),
    HumanMessage("¿Qué sabés del personaje Sandoval y qué escena lo define?"),
])

display(Markdown("###  Respuesta del LLM (ANTES de ejecutar tools)"))
print(f"Contenido textual: '{respuesta.content}'")
print(f"Tool calls generados: {respuesta.tool_calls}")
print()


2026-03-05 19:09:00,056 - SecretoOjosA2A - INFO - [CONECTAR!!] Tools vinculados al LLM


2026-03-05 19:09:02,513 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


###  Respuesta del LLM (ANTES de ejecutar tools)

Contenido textual: ''
Tool calls generados: [{'name': 'consultar_personaje', 'args': {'nombre': 'Sandoval'}, 'id': 'call_njIggVtuIurjUcvACgDURHTD', 'type': 'tool_call'}, {'name': 'consultar_escena', 'args': {'titulo': 'La entrega del informe sobre el caso'}, 'id': 'call_kFdgnC3rFQaEvh1BlUm9CoHC', 'type': 'tool_call'}, {'name': 'consultar_escena', 'args': {'titulo': 'La confesión de Morales'}, 'id': 'call_9NA56PShRCd14M8TsroV6DmW', 'type': 'tool_call'}]



In [26]:
# Ejecutar los tool calls y devolver resultados al LLM
messages = [
    SystemMessage("Eres un analista de El Secreto de sus Ojos. Usá las herramientas disponibles."),
    HumanMessage("¿Qué sabés del personaje Sandoval y qué escena lo define?"),
    respuesta,  # AIMessage con los tool_calls
]

tools_por_nombre = {t.name: t for t in tools}

for tc in respuesta.tool_calls:
    resultado = tools_por_nombre[tc["name"]].invoke(tc["args"])
    messages.append(ToolMessage(content=resultado, tool_call_id=tc["id"]))
    logger.info(f"[TOOL] {tc['name']}({tc['args']})")

# El LLM ahora sí tiene los resultados y puede responder
respuesta_final = llm_con_tools.invoke(messages)
display(Markdown(respuesta_final.content))

2026-03-05 19:11:36,523 - SecretoOjosA2A - INFO - [TOOL] consultar_personaje({'nombre': 'Sandoval'})
2026-03-05 19:11:36,526 - SecretoOjosA2A - INFO - [TOOL] consultar_escena({'titulo': 'La entrega del informe sobre el caso'})
2026-03-05 19:11:36,528 - SecretoOjosA2A - INFO - [TOOL] consultar_escena({'titulo': 'La confesión de Morales'})
2026-03-05 19:11:38,108 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
